# Ion Channel Pore Profile Analysis (HOLE)

Measures the **pore radius profile** along the conduction pathway of an ion
channel, using [HOLE](https://www.holeprogram.org/) through the `mdahole2`
interface to MDAnalysis.

The notebook produces:

| Output | What it is |
|---|---|
| Pore radius profile of a reference structure | Radius vs position along the pore for a single PDB |
| Pore profiles averaged over trajectories | One mean profile per simulation condition |
| Aligned (registered) profiles | Profiles shifted onto a common axis so conditions are comparable |
| Constriction table | Radius and location of the narrowest points, with domain annotation |
| Pore surface PDB | A ring-of-pseudoatoms surface for viewing the lumen in PyMOL or VMD |

The narrowest point of the profile is the physically interesting number: it is
what determines whether a hydrated ion can pass, and how it shifts between
conditions is the gating signal this notebook is built to expose.

---

## Requirements

`mdahole2` is a Python wrapper. It calls the **external HOLE executable**, which
is separately licensed and must be installed and on your `PATH`:

```bash
pip install mdahole2
# then install HOLE itself from https://www.holeprogram.org/ and ensure
# the `hole` binary is on PATH, or set HOLE_EXECUTABLE in the configuration cell
```

The notebook checks for the executable before doing any work and tells you
clearly if it is missing.

**No HOLE installed?** Leave `USE_DEMO_DATA = True`. The notebook then generates
synthetic pore profiles and exercises the alignment, constriction-finding,
plotting and PDB-writing code without calling HOLE at all. This is a useful way
to check the pipeline, but the profiles are invented and carry no physical
meaning.

## Expected data layout

```
<DATA_ROOT>/
├── reference/
│   └── channel_reference.pdb      # single structure, e.g. a repaired PDB entry
├── coarse/
│   └── 305K05/ ...
└── atomistic/
    ├── 305K05/
    │   ├── NPTProteinCenter.pdb   # topology
    │   └── NPTProteinCenter.xtc   # trajectory, protein centred
    └── ...
```

## Generating the input files

HOLE needs the channel centred and its axis roughly along z:

```bash
# Centre the protein and remove periodic jumps
gmx trjconv -s npt.tpr -f npt.xtc -o NPTProteinCenter.xtc \
            -pbc mol -center -n index.ndx

# Matching single-frame topology
gmx trjconv -s npt.tpr -f NPTProteinCenter.xtc -o NPTProteinCenter.pdb -dump 0
```

If your channel axis is not along z, set `HOLE_CVECT` in the configuration cell
to the pore axis vector, or `HOLE_CPOINT` to a point inside the lumen.

## Units

HOLE works in **ångström** throughout, so unlike the other notebooks in this
repository there is no nm conversion. Positions along the pore are reported on
HOLE's reaction-coordinate axis, in Å.

## Adapting to a different channel — checklist

1. `N_PROTOMERS`, `RESIDUE_OFFSET`, `DOMAIN_SEGMENTS` — as in the other notebooks.
2. `HOLE_CVECT` / `HOLE_CPOINT` — only if the pore is not aligned with z.
3. `HOLE_END_RADIUS` — the radius at which HOLE decides it has left the pore.
   Too large and the profile runs off into bulk solvent; too small and it stops
   short of the vestibules.
4. `PORE_AXIS_WINDOW` — the span of the reaction coordinate to keep for plotting.


---

## 1. Imports

In [ ]:
# ---------------------------------------------------------------------------
# Standard library
# ---------------------------------------------------------------------------
import os
import re
import shutil
import warnings
import itertools
from dataclasses import dataclass
from pathlib import Path

# ---------------------------------------------------------------------------
# Third party
# ---------------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch                 # domain legend handles
from matplotlib.ticker import ScalarFormatter        # compact axis labels

import MDAnalysis as mda                             # trajectory handling

from scipy.interpolate import interp1d               # resample profiles onto a common axis
from scipy.optimize import minimize_scalar           # align profiles by shifting
from scipy.signal import find_peaks                  # locate constrictions

# mdahole2 wraps the external HOLE program. Imported lazily below so that the
# notebook still runs in demo mode on a machine without HOLE installed.
try:
    from mdahole2.analysis import HoleAnalysis
    HOLE_IMPORT_ERROR = None
except Exception as exc:                             # pragma: no cover
    HoleAnalysis = None
    HOLE_IMPORT_ERROR = exc

# ---------------------------------------------------------------------------
# Shared repository configuration
# ---------------------------------------------------------------------------
# channel_config.py sits beside this notebook and holds everything common to
# every notebook here: the data root, the simulation grid, the channel topology,
# the domain boundaries and the plotting primitives. The star import brings in
# only the shared, read-only names it lists in __all__.
import channel_config as cc
from channel_config import *

%matplotlib inline

print(f"numpy {np.__version__}")
print(f"MDAnalysis {mda.__version__}")
print("mdahole2 import:", "ok" if HoleAnalysis is not None else f"FAILED ({HOLE_IMPORT_ERROR})")

---

## 2. Configuration

Everything shared with the other notebooks lives in `channel_config.py`; only this notebook's own settings are here.

In [ ]:
# ===========================================================================
#                            C O N F I G U R A T I O N
#
#  Only the settings specific to THIS notebook live here. Everything shared
#  across the repository - the data root, the simulation grid, the channel
#  topology, the domain boundaries and the plot style - lives in
#  channel_config.py, so it can only ever be defined once.
#
#  To change a shared value (protomer count, temperatures, domains, ...), edit
#  channel_config.py and every notebook picks it up.
# ===========================================================================

# ---------------------------------------------------------------------------
# 1. This notebook
# ---------------------------------------------------------------------------
# Set USE_DEMO_DATA = True to generate small synthetic inputs and run the whole
# notebook without real trajectories. Set it to False to use your own data;
# point channel_config.REAL_DATA_ROOT at it, or set the MD_DATA_ROOT
# environment variable.
USE_DEMO_DATA = True

# Names this notebook's output directory, results/<NOTEBOOK_SLUG>/. Change it if
# you copy this notebook as the basis for a new analysis, so the two cannot
# overwrite each other's figures.
NOTEBOOK_SLUG = "protein_channel_pore"

# Write every figure to OUTPUT_DIR as it is drawn.
SAVE_FIGURES = False

# Logical name -> file name inside each condition directory.
FILENAMES = {
    "topology": "NPTProteinCenter.pdb",
    "trajectory": "NPTProteinCenter.xtc",
}

# Single reference structure, analysed on its own as a baseline. Relative to
# DATA_ROOT; set to None to skip the reference-structure section.
REFERENCE_STRUCTURE = Path("reference") / "channel_reference.pdb"

# Resolved paths.
DATA_ROOT = cc.resolve_data_root(USE_DEMO_DATA)
DEMO_DATA_ROOT = cc.DEMO_DATA_ROOT
OUTPUT_DIR = cc.make_output_dir(NOTEBOOK_SLUG)


def build_path(model_level, temperature_k, salinity_mm, key):
    """Return the full path to one input file.

    Override this if your directory tree is shaped differently, e.g. if the
    model level is a filename suffix rather than a parent directory. The
    condition directory name itself comes from cc.condition_dirname().
    """
    return (
        DATA_ROOT
        / model_level
        / cc.condition_dirname(temperature_k, salinity_mm)
        / FILENAMES[key]
    )


def save_figure(fig, name):
    """Save a figure into this notebook's output directory."""
    return cc.save_figure(fig, name, OUTPUT_DIR, SAVE_FIGURES)


# ---------------------------------------------------------------------------
# 2. HOLE settings
# ---------------------------------------------------------------------------
# Path to the external HOLE executable. Leave as "hole" to use whatever is on
# PATH, or give an absolute path.
HOLE_EXECUTABLE = os.environ.get("HOLE_EXECUTABLE", "hole")

# Radius (A) at which HOLE decides it has left the pore and stops. HOLE's own
# default is 5.0; a larger value follows the pathway further into the vestibules
# at the cost of sometimes escaping into bulk solvent.
HOLE_END_RADIUS = 5.0

# Pore axis. Leave both as None when the channel axis is already along z, which
# is what `gmx trjconv -center` on a membrane system normally gives.
#   HOLE_CVECT  - direction of the pore axis, e.g. [0, 0, 1]
#   HOLE_CPOINT - a point known to lie inside the lumen, e.g. [0, 0, 0]
HOLE_CVECT = None
HOLE_CPOINT = None

# Frames to analyse. HOLE is expensive - one call per frame - so a stride of
# 100 or more is normal for a long trajectory.
# The demo trajectories are short; a real-run stride would leave nothing.
FRAME_STEP = 1 if USE_DEMO_DATA else 100
START_FRAME = 0
END_FRAME = None          # None -> to the end of each trajectory

# ---------------------------------------------------------------------------
# 3. Profile post-processing
# ---------------------------------------------------------------------------
# Reaction-coordinate window (A) to keep when plotting. None -> keep everything
# the profiles have in common.
PORE_AXIS_WINDOW = None

# Number of points on the common axis that profiles are resampled onto.
PROFILE_N_POINTS = 400

# Minimum depth (A) a dip must have to count as a constriction, and the minimum
# separation (A) between reported constrictions.
CONSTRICTION_PROMINENCE = 0.5
CONSTRICTION_MIN_SEPARATION = 3.0


# ---------------------------------------------------------------------------
# Apply the shared plot style and report the resolved configuration
# ---------------------------------------------------------------------------
cc.apply_style()
cc.describe_config(DATA_ROOT, USE_DEMO_DATA, OUTPUT_DIR)

---

## 3. The condition grid

In [ ]:
# The set of simulations, built once from the shared grid definition. Every
# later selection is a query against it - grid.where(salinity_mm=50) returns the
# positions of those runs - so slicing is never hard coded as [0:6] or [12:18].
grid = cc.ConditionGrid(TEMPERATURES_K, SALINITIES_MM, build_path)
grid.describe()

# Convenience: the reference structure path, resolved against DATA_ROOT.
reference_structure_path = (
    None if REFERENCE_STRUCTURE is None else DATA_ROOT / REFERENCE_STRUCTURE
)
print(f"\nReference structure: {reference_structure_path}")

---

## 4. Demo data (optional)

HOLE is an external, separately licensed program, so demo mode does not call it. It synthesises pore profiles directly and feeds them through the same alignment, constriction-finding and plotting code.

In [ ]:
# ===========================================================================
#  Optional: synthetic demo data
#
#  Builds a small synthetic system and, more importantly, synthetic *pore
#  profiles*. HOLE is an external program that cannot be assumed present, so in
#  demo mode the notebook bypasses it entirely and feeds invented profiles into
#  the same alignment, constriction-finding and plotting code.
#
#  The profiles have the qualitative shape of a real channel - wide vestibules
#  at both ends, a selectivity-filter constriction near one end and a gate
#  constriction near the other - but the numbers are fabricated.
# ===========================================================================

DEMO_N_ATOMS_PER_RESIDUE = 1
DEMO_RESIDUES_PER_PROTOMER = 60      # kept small; HOLE is not actually run
DEMO_N_FRAMES = 5
DEMO_PORE_HALF_LENGTH = 45.0         # A, half the span of the reaction coordinate


def synthetic_pore_profile(rng, temperature_k, salinity_mm, n_points=None):
    """Return (reaction_coordinate, radius) for one invented pore profile.

    A funnel-shaped baseline - wide vestibules at both ends, narrower in the
    middle - with two constrictions cut into it: a fixed one standing in for a
    selectivity filter, and a temperature-dependent one standing in for a gate.
    The gate widens with temperature so the demo plots show a trend; that is
    written into the formula, not discovered from any simulation.
    """
    n_points = PROFILE_N_POINTS if n_points is None else n_points
    heat = (temperature_k - min(TEMPERATURES_K)) / 25.0

    # Each run gets a random offset along the axis, so the alignment step below
    # has a real misregistration to correct.
    offset = rng.uniform(-6.0, 6.0)
    centred = np.linspace(-DEMO_PORE_HALF_LENGTH, DEMO_PORE_HALF_LENGTH, n_points)
    z = centred + offset

    # Vestibules open out towards both ends.
    baseline = 4.5 + 5.5 * (np.abs(centred) / DEMO_PORE_HALF_LENGTH) ** 2

    # Fixed narrow constriction (selectivity-filter-like), floor around 1.2 A.
    selectivity_filter = 3.3 * np.exp(-0.5 * ((centred - 12.0) / 3.0) ** 2)

    # Gate-like constriction that widens with temperature and, weakly, salinity.
    gate_depth = 4.3 - 1.8 * heat - 0.2 * (salinity_mm / 150.0)
    gate = gate_depth * np.exp(-0.5 * ((centred + 14.0) / 4.5) ** 2)

    radius = baseline - selectivity_filter - gate
    radius += rng.normal(0.0, 0.04, n_points)
    return z, np.clip(radius, 0.15, None)


def generate_demo_structures():
    """Write a minimal topology/trajectory pair per condition, plus a reference.

    These exist so that the file-discovery and validation code paths are
    exercised. They are far too coarse for a real HOLE calculation.
    """
    rng = np.random.default_rng(0)
    n_res = DEMO_RESIDUES_PER_PROTOMER * N_PROTOMERS
    n_atoms = n_res * DEMO_N_ATOMS_PER_RESIDUE

    universe = mda.Universe.empty(
        n_atoms, n_residues=n_res,
        atom_resindex=np.repeat(np.arange(n_res), DEMO_N_ATOMS_PER_RESIDUE),
        trajectory=True,
    )
    universe.add_TopologyAttr("name", ["CA"] * n_atoms)
    universe.add_TopologyAttr("type", ["C"] * n_atoms)
    universe.add_TopologyAttr("resname", ["ALA"] * n_res)
    # Chosen so that adding RESIDUE_OFFSET puts each model level onto the
    # common numbering scheme, as real data from either resolution would.
    first_residue = RESIDUE_NUMBER_ORIGIN - min(RESIDUE_OFFSET.values())
    universe.add_TopologyAttr(
        "resid", list(range(first_residue, first_residue + n_res)))
    universe.add_TopologyAttr("segid", ["A"])
    universe.dimensions = [120.0, 120.0, 120.0, 90.0, 90.0, 90.0]

    def positions_for(seed):
        local = np.random.default_rng(seed)
        # A hollow cylinder along z, so the geometry is at least pore-like.
        angle = local.uniform(0, 2 * np.pi, n_atoms)
        z = local.uniform(-DEMO_PORE_HALF_LENGTH, DEMO_PORE_HALF_LENGTH, n_atoms)
        radius = local.uniform(12.0, 16.0, n_atoms)
        return np.column_stack([
            radius * np.cos(angle) + 60.0,
            radius * np.sin(angle) + 60.0,
            z + 60.0,
        ])

    written = 0
    for model_level in MODEL_LEVELS:
        for cond in grid:
            topology = build_path(model_level, cond.temperature_k,
                                  cond.salinity_mm, "topology")
            trajectory = build_path(model_level, cond.temperature_k,
                                    cond.salinity_mm, "trajectory")
            topology.parent.mkdir(parents=True, exist_ok=True)
            universe.atoms.positions = positions_for(rng.integers(1 << 30))
            # The PDB writer warns about topology attributes a synthetic
            # Universe legitimately lacks (altLocs, icodes, ...). Silence just
            # those, rather than filtering warnings globally.
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", UserWarning)
                universe.atoms.write(str(topology))
            with mda.Writer(str(trajectory), n_atoms=n_atoms) as writer:
                for frame in range(DEMO_N_FRAMES):
                    universe.atoms.positions = positions_for(frame)
                    writer.write(universe.atoms)
            written += 2

    if reference_structure_path is not None:
        reference_structure_path.parent.mkdir(parents=True, exist_ok=True)
        universe.atoms.positions = positions_for(99)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            universe.atoms.write(str(reference_structure_path))
        written += 1
    return written


if USE_DEMO_DATA:
    n_written = generate_demo_structures()
    print(f"Wrote {n_written} synthetic structure/trajectory files under "
          f"{DEMO_DATA_ROOT.resolve()}")
    print("Pore profiles will be synthesised directly; HOLE is not called.")
    print("\nThese files are for testing the pipeline only - not simulation data.")
else:
    for model_level in MODEL_LEVELS:
        for key in ("topology", "trajectory"):
            missing = grid.report_missing(model_level, key)
            if missing:
                print(f"  WARNING {model_level}/{key}: {len(missing)} file(s) missing, "
                      f"first is {missing[0]}")
    if reference_structure_path is not None and not reference_structure_path.is_file():
        print(f"  WARNING reference structure not found: {reference_structure_path}")

---

## 5. Pore profiling

In [ ]:
# ===========================================================================
#  Pore profiling
#
#  One HOLE run per frame gives a radius as a function of position along the
#  pore. Averaging those into a single profile per condition needs care, because
#  HOLE returns a *different* set of sample points for every frame: the pathway
#  wanders, and the profile stops wherever the radius exceeds HOLE_END_RADIUS.
#
#  This implementation resamples every frame onto one common axis before
#  averaging. The original approach of truncating each profile to the shortest
#  length and averaging element-wise silently compares different positions along
#  the pore to each other whenever the sample spacing differs between frames.
# ===========================================================================

def hole_is_available(executable=None):
    """Return True when the external HOLE binary can be found."""
    executable = HOLE_EXECUTABLE if executable is None else executable
    if HoleAnalysis is None:
        return False
    if os.path.sep in str(executable):
        return Path(executable).is_file()
    return shutil.which(str(executable)) is not None


def resample_profiles(coordinates, radii, n_points=None, axis_window=None):
    """Resample a set of (coordinate, radius) profiles onto one common axis.

    Only the range covered by *every* profile is used, so no value is
    extrapolated. Returns (common_axis, stacked_radii) where `stacked_radii` has
    shape (n_profiles, n_points).
    """
    n_points = PROFILE_N_POINTS if n_points is None else n_points
    coordinates = [np.asarray(c, dtype=float) for c in coordinates]
    radii = [np.asarray(r, dtype=float) for r in radii]

    lo = max(c.min() for c in coordinates)
    hi = min(c.max() for c in coordinates)
    if axis_window is not None:
        lo = max(lo, axis_window[0])
        hi = min(hi, axis_window[1])
    if not np.isfinite([lo, hi]).all() or hi <= lo:
        raise ValueError(
            "The profiles share no common range of the reaction coordinate. "
            "This usually means the pore axis drifted between frames - check "
            "that the protein was centred, or set HOLE_CVECT/HOLE_CPOINT."
        )

    common_axis = np.linspace(lo, hi, n_points)
    stacked = np.empty((len(radii), n_points), dtype=float)
    for i, (coord, radius) in enumerate(zip(coordinates, radii)):
        order = np.argsort(coord)          # interp1d needs a sorted x
        stacked[i] = interp1d(coord[order], radius[order], kind="linear",
                              bounds_error=False, fill_value=np.nan)(common_axis)
    return common_axis, stacked


class PoreProfileAnalysis:
    """Runs HOLE over structures or trajectories and averages the profiles.

    Parameters
    ----------
    topologies : sequence of path
        One topology per simulation.
    trajectories : sequence of path or None
        Matching trajectories. Pass None to analyse each topology as a single
        static structure.
    labels : sequence of str, optional
        One label per simulation, used for plot legends and error messages.
    frame_step, start_frame, end_frame :
        Frame selection, defaulting to the configuration cell.

    Attributes (after `run()`)
    --------------------------
    axis : list of np.ndarray
        Common reaction coordinate per simulation, in angstrom.
    radius : list of np.ndarray
        Mean pore radius per simulation, in angstrom.
    radius_std : list of np.ndarray
        Standard deviation across frames, or zeros for a static structure.
    """

    TOPOLOGY_SUFFIXES = {".pdb", ".gro"}
    TRAJECTORY_SUFFIXES = {".xtc", ".trr", ".dcd"}

    def __init__(self, topologies, trajectories=None, labels=None,
                 frame_step=None, start_frame=None, end_frame=None):
        # Accept a bare path as a one-element list. The original code type
        # checked for a string and then iterated over its characters, so a
        # single file could never actually be analysed.
        if isinstance(topologies, (str, Path)):
            topologies = [topologies]
        self.topologies = [Path(p) for p in topologies]
        if not self.topologies:
            raise ValueError("No topologies given.")

        bad = [p for p in self.topologies
               if p.suffix.lower() not in self.TOPOLOGY_SUFFIXES]
        if bad:
            raise ValueError(
                f"Unsupported topology type '{bad[0].suffix}'. "
                f"Supported: {sorted(self.TOPOLOGY_SUFFIXES)}")

        if trajectories is None:
            self.trajectories = [None] * len(self.topologies)
        else:
            if isinstance(trajectories, (str, Path)):
                trajectories = [trajectories]
            self.trajectories = [None if t is None else Path(t) for t in trajectories]
            bad = [p for p in self.trajectories
                   if p is not None and p.suffix.lower() not in self.TRAJECTORY_SUFFIXES]
            if bad:
                raise ValueError(
                    f"Unsupported trajectory type '{bad[0].suffix}'. "
                    f"Supported: {sorted(self.TRAJECTORY_SUFFIXES)}")

        if len(self.topologies) != len(self.trajectories):
            raise ValueError(
                f"Got {len(self.topologies)} topologies but "
                f"{len(self.trajectories)} trajectories; they must correspond.")

        missing = [p for p in self.topologies + self.trajectories
                   if p is not None and not p.is_file()]
        if missing:
            raise FileNotFoundError(
                f"{len(missing)} input file(s) not found. First: {missing[0]}\n"
                "Check DATA_ROOT and build_path() in the configuration cell.")

        self.labels = (list(labels) if labels is not None
                       else [p.parent.name for p in self.topologies])
        self.frame_step = FRAME_STEP if frame_step is None else frame_step
        self.start_frame = START_FRAME if start_frame is None else start_frame
        self.end_frame = END_FRAME if end_frame is None else end_frame

        self.axis = None
        self.radius = None
        self.radius_std = None

    # -- HOLE keyword assembly ---------------------------------------------
    def _hole_kwargs(self):
        kwargs = {"executable": HOLE_EXECUTABLE, "end_radius": HOLE_END_RADIUS}
        if HOLE_CVECT is not None:
            kwargs["cvect"] = list(HOLE_CVECT)
        if HOLE_CPOINT is not None:
            kwargs["cpoint"] = list(HOLE_CPOINT)
        return kwargs

    # -- main entry point ---------------------------------------------------
    def run(self):
        """Profile every simulation. Returns self."""
        if not hole_is_available():
            raise RuntimeError(
                f"The HOLE executable ('{HOLE_EXECUTABLE}') was not found.\n"
                "HOLE is a separately licensed external program: install it from "
                "https://www.holeprogram.org/ and put it on PATH, or set "
                "HOLE_EXECUTABLE in the configuration cell.\n"
                "To try the rest of the notebook without HOLE, set "
                "USE_DEMO_DATA = True.")

        self.axis, self.radius, self.radius_std = [], [], []
        for topology, trajectory, label in zip(self.topologies, self.trajectories,
                                               self.labels):
            if trajectory is None:
                universe = mda.Universe(str(topology))
                with HoleAnalysis(universe, **self._hole_kwargs()) as analysis:
                    analysis.run()
                profiles = list(analysis.results["profiles"].values())
            else:
                universe = mda.Universe(str(topology), str(trajectory))
                # end_frame is resolved per universe. The original computed it
                # once and reused it, so a stop value taken from the first
                # trajectory silently applied to every later one.
                stop = (len(universe.trajectory) if self.end_frame is None
                        else self.end_frame)
                with HoleAnalysis(universe, **self._hole_kwargs()) as analysis:
                    analysis.run(start=self.start_frame, stop=stop,
                                 step=self.frame_step)
                profiles = list(analysis.results["profiles"].values())

            coordinates, radii = [], []
            for profile in profiles:
                coord = np.asarray(profile["rxn_coord"], dtype=float)
                radius = np.asarray(profile["radius"], dtype=float)
                if coord.size > 1 and radius.size == coord.size:
                    coordinates.append(coord)
                    radii.append(radius)
            if not radii:
                raise RuntimeError(
                    f"HOLE returned no usable profile for '{label}' "
                    f"({topology.name}). The pore may not be aligned with the "
                    "search vector - try setting HOLE_CVECT or HOLE_CPOINT.")

            common_axis, stacked = resample_profiles(coordinates, radii,
                                                     axis_window=PORE_AXIS_WINDOW)
            self.axis.append(common_axis)
            self.radius.append(np.nanmean(stacked, axis=0))
            self.radius_std.append(np.nanstd(stacked, axis=0))
        return self

    def load_precomputed(self, axis, radius, radius_std=None):
        """Populate the results directly, bypassing HOLE.

        Used by the demo path, and useful for re-plotting profiles saved from an
        earlier run without paying for the HOLE calculation again.
        """
        self.axis = [np.asarray(a, dtype=float) for a in axis]
        self.radius = [np.asarray(r, dtype=float) for r in radius]
        self.radius_std = ([np.zeros_like(r) for r in self.radius]
                           if radius_std is None
                           else [np.asarray(s, dtype=float) for s in radius_std])
        return self

    # -- reporting ----------------------------------------------------------
    def minimum_radius(self):
        """Return the narrowest radius of each profile, in angstrom."""
        return np.array([np.nanmin(r) for r in self.radius])

    def constrictions(self, prominence=None, min_separation=None):
        """Locate the constrictions (local radius minima) in each profile.

        Returns a list of (positions, radii) tuples, one per simulation, ordered
        by position along the pore. `min_separation` is given in angstrom and
        converted to samples using each profile's own spacing.
        """
        prominence = (CONSTRICTION_PROMINENCE if prominence is None else prominence)
        min_separation = (CONSTRICTION_MIN_SEPARATION if min_separation is None
                          else min_separation)
        results = []
        for axis, radius in zip(self.axis, self.radius):
            spacing = float(np.mean(np.diff(axis))) if axis.size > 1 else 1.0
            distance = max(1, int(round(min_separation / max(spacing, 1e-9))))
            # find_peaks on the negated radius finds minima.
            peaks, _ = find_peaks(-radius, prominence=prominence, distance=distance)
            results.append((axis[peaks], radius[peaks]))
        return results

---

## 6. Aligning and exporting profiles

In [ ]:
# ===========================================================================
#  Aligning profiles onto a common axis
#
#  HOLE's reaction coordinate is measured from wherever the pathway search
#  happened to start, so the same physical constriction can appear at different
#  coordinates in different runs. Comparing conditions requires registering the
#  profiles first, by shifting each one so that its constrictions line up with
#  a consensus set.
# ===========================================================================

def align_profiles(axes, radii, prominence=None, max_shift=None):
    """Shift each profile along its axis so that constrictions coincide.

    The reference is the mean position of the constrictions across all profiles.
    Each profile is then shifted by the amount that minimises the squared
    distance between its constrictions and that reference.

    Returns (shifted_axes, shifts). `shifts` is in angstrom and is reported so
    that an implausibly large correction is visible rather than silent.
    """
    prominence = (CONSTRICTION_PROMINENCE if prominence is None else prominence)

    minima_positions = []
    for axis, radius in zip(axes, radii):
        axis = np.asarray(axis, dtype=float)
        radius = np.asarray(radius, dtype=float)
        peaks, _ = find_peaks(-radius, prominence=prominence)
        minima_positions.append(axis[peaks])

    n_found = [len(m) for m in minima_positions]
    if min(n_found) == 0:
        raise ValueError(
            "At least one profile has no detectable constriction, so there is "
            f"nothing to align on (constrictions found per profile: {n_found}). "
            "Lower CONSTRICTION_PROMINENCE.")

    # Consensus positions: mean over profiles, padding with NaN where a profile
    # has fewer constrictions than the richest one.
    max_len = max(n_found)
    padded = np.full((len(minima_positions), max_len), np.nan)
    for i, positions in enumerate(minima_positions):
        padded[i, :len(positions)] = positions
    reference = np.nanmean(padded, axis=0)

    shifted_axes, shifts = [], []
    for axis, positions in zip(axes, minima_positions):
        axis = np.asarray(axis, dtype=float)
        n_compare = min(len(positions), len(reference))
        target = reference[:n_compare]
        observed = positions[:n_compare]

        def cost(shift, observed=observed, target=target):
            return float(np.sum((observed + shift - target) ** 2))

        span = float(axis.max() - axis.min())
        bound = span if max_shift is None else float(max_shift)
        shift = float(minimize_scalar(cost, bounds=(-bound, bound),
                                      method="bounded").x)
        shifted_axes.append(axis + shift)
        shifts.append(shift)
    return shifted_axes, np.array(shifts)


def write_pore_surface_pdb(axis, radius, output_pdb, n_theta=48, step=4,
                           residue_name="POR", chain="A"):
    """Write a ring-of-pseudoatoms surface tracing the pore lumen.

    At each sampled position along the pore a ring of `n_theta` carbon
    pseudoatoms is placed at the local radius, giving a mesh that can be loaded
    alongside the real structure to see the conduction pathway:

        load pore_surface.pdb
        show surface, pore_surface

    `step` decimates the axis (every 4th point by default) to keep the file
    small. Note the output is a visualisation aid, not a physical model - the
    atoms have no chemistry.

    Returns the number of pseudoatoms written.
    """
    axis = np.asarray(axis, dtype=float).ravel()
    radius = np.asarray(radius, dtype=float).ravel()
    if axis.size != radius.size:
        raise ValueError(
            f"axis has {axis.size} points but radius has {radius.size}; "
            "they must correspond one to one.")

    output_pdb = Path(output_pdb)
    output_pdb.parent.mkdir(parents=True, exist_ok=True)

    theta = np.linspace(0.0, 2.0 * np.pi, n_theta, endpoint=False)
    serial = 0
    with open(output_pdb, "w") as handle:
        handle.write("REMARK   Pore surface mesh generated for visualisation only.\n")
        for residue_index, (z, r) in enumerate(zip(axis[::step], radius[::step]), start=1):
            for angle in theta:
                serial += 1
                # PDB columns allow 5 digits of serial and 4 of residue id;
                # wrap rather than emit a malformed line on a long pore.
                atom_serial = serial % 100000
                residue_id = residue_index % 10000
                handle.write(
                    f"ATOM  {atom_serial:5d}  C   {residue_name:>3s} {chain:1s}"
                    f"{residue_id:4d}    "
                    f"{float(r * np.cos(angle)):8.3f}"
                    f"{float(r * np.sin(angle)):8.3f}"
                    f"{float(z):8.3f}"
                    f"{1.00:6.2f}{float(r):6.2f}          C\n"
                )
        handle.write("END\n")
    print(f"  {output_pdb}: {serial} pseudoatoms "
          f"({len(axis[::step])} rings of {n_theta})")
    return serial

---

## 7. Pore plotting

In [ ]:
# ===========================================================================
#  Pore-specific plotting
# ===========================================================================

def plot_pore_profiles(axes, radii, labels=None, stds=None, colors=None,
                       legend_title=None, title=None, vertical=True,
                       show_band=False, figure_name=None):
    """Plot pore radius against position along the pore.

    `vertical=True` puts the pore axis on the y axis and the radius on the x
    axis, which matches how a channel is usually drawn: the membrane normal runs
    up the page. Set it to False for a conventional radius-vs-position plot.

    `show_band=True` shades one standard deviation across frames, which is the
    honest way to show how much a profile moves during a trajectory.
    """
    figsize = (8, 15) if vertical else (15, 8)
    fig, ax = plt.subplots(figsize=figsize)
    colors = resolve_colors(colors, len(radii))
    labels = labels if labels is not None else [None] * len(radii)
    stds = stds if stds is not None else [None] * len(radii)

    for axis, radius, std, color, label in zip(axes, radii, stds, colors, labels):
        if vertical:
            ax.plot(radius, axis, color=color, label=label, linewidth=2)
            if show_band and std is not None:
                ax.fill_betweenx(axis, radius - std, radius + std,
                                 color=color, alpha=0.2, linewidth=0)
        else:
            ax.plot(axis, radius, color=color, label=label, linewidth=2)
            if show_band and std is not None:
                ax.fill_between(axis, radius - std, radius + std,
                                color=color, alpha=0.2, linewidth=0)

    radius_label = r"Pore Radius ($\mathrm{\AA}$)"
    axis_label = r"Position Along Pore ($\mathrm{\AA}$)"
    axis_lo = min(float(np.min(a)) for a in axes)
    axis_hi = max(float(np.max(a)) for a in axes)
    radius_lo = 0.0
    radius_hi = max(float(np.max(r)) for r in radii) * 1.05

    if vertical:
        ax.set_xlabel(radius_label)
        ax.set_ylabel(axis_label)
        ax.set_xlim(radius_lo, radius_hi)
        ax.set_ylim(axis_lo, axis_hi)
        apply_ticks(ax, "x", radius_lo, radius_hi, n_ticks=7)
        apply_ticks(ax, "y", axis_lo, axis_hi, n_ticks=10)
    else:
        ax.set_xlabel(axis_label)
        ax.set_ylabel(radius_label)
        ax.set_xlim(axis_lo, axis_hi)
        ax.set_ylim(radius_lo, radius_hi)
        apply_ticks(ax, "x", axis_lo, axis_hi, n_ticks=10)
        apply_ticks(ax, "y", radius_lo, radius_hi, n_ticks=7)

    if title:
        ax.set_title(title, fontsize=FONTSIZE)
    if any(l is not None for l in labels):
        ax.legend(title=legend_title, loc="lower left",
                  title_fontsize=FONTSIZE * 0.8, fontsize=FONTSIZE * 0.8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if figure_name:
        save_figure(fig, figure_name)
    return fig, ax


def plot_minimum_radius(values_by_group, group_labels, x_values, xlabel,
                        colors=None, title=None, figure_name=None):
    """Plot the narrowest pore radius against a scanned variable.

    This is the summary figure: one line per group, showing how the limiting
    constriction responds to temperature (or whatever `x_values` encodes).
    """
    fig, ax = plt.subplots(figsize=(15, 8))
    colors = resolve_colors(colors, len(values_by_group))
    for values, label, color in zip(values_by_group, group_labels, colors):
        ax.plot(x_values, values, color=color, label=label, marker="o",
                markersize=9, linewidth=2)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(r"Minimum Pore Radius ($\mathrm{\AA}$)")
    if title:
        ax.set_title(title, fontsize=FONTSIZE)
    ax.legend(loc="best", fontsize=FONTSIZE * 0.8)
    all_values = np.concatenate([np.asarray(v, dtype=float) for v in values_by_group])
    apply_ticks(ax, "y", float(all_values.min()) * 0.95,
                float(all_values.max()) * 1.05, n_ticks=7)
    ax.set_xticks(list(x_values))
    ax.set_xticklabels([f"{x:g}" for x in x_values])
    ax.margins(x=0.05)

    if figure_name:
        save_figure(fig, figure_name)
    return fig, ax

---

# Analysis

Everything above defines the pipeline; everything below runs it.

---

## 8. Reference structure

In [ ]:
# Pore profile of the single reference structure, and a surface mesh for viewing.
#
# In demo mode the profile is synthesised; otherwise HOLE is run on the
# structure. Either way the downstream code is identical.
reference_profile = None

if reference_structure_path is None:
    print("REFERENCE_STRUCTURE is None - skipping.")
elif USE_DEMO_DATA:
    rng = np.random.default_rng(12345)
    axis, radius = synthetic_pore_profile(rng, temperature_k=min(TEMPERATURES_K),
                                          salinity_mm=min(SALINITIES_MM))
    reference_profile = PoreProfileAnalysis(
        reference_structure_path, labels=["reference"]
    ).load_precomputed([axis], [radius])
    print("Synthesised a reference profile (HOLE not called).")
elif not hole_is_available():
    print(f"HOLE executable '{HOLE_EXECUTABLE}' not found - skipping the "
          "reference structure. See the requirements note at the top.")
else:
    reference_profile = PoreProfileAnalysis(
        reference_structure_path, labels=["reference"]
    ).run()

if reference_profile is not None:
    minimum = reference_profile.minimum_radius()[0]
    print(f"Narrowest radius in the reference structure: {minimum:.2f} A")

    positions, radii_at_min = reference_profile.constrictions()[0]
    print(f"\n{len(positions)} constriction(s) detected:")
    for position, radius_value in zip(positions, radii_at_min):
        print(f"  at {position:+8.2f} A along the pore: radius {radius_value:5.2f} A")

    write_pore_surface_pdb(
        reference_profile.axis[0], reference_profile.radius[0],
        OUTPUT_DIR / "pore_surface_reference.pdb",
    )

---

## 9. Profiles over the condition grid

In [ ]:
# Pore profiles for every simulation condition, at both model levels.
#
# With real data this is the expensive cell: HOLE is called once per frame per
# condition. Raise FRAME_STEP if it takes too long.
pore = {}

for model_level in MODEL_LEVELS:
    print(f"\n{model_level}:")
    analysis = PoreProfileAnalysis(
        topologies=grid.paths(model_level, "topology"),
        trajectories=grid.paths(model_level, "trajectory"),
        labels=grid.labels(by="both"),
    )

    if USE_DEMO_DATA:
        # Synthesise one profile per condition instead of calling HOLE.
        rng = np.random.default_rng(abs(hash(model_level)) % (1 << 32))
        axes, radii = [], []
        for cond in grid:
            axis, radius = synthetic_pore_profile(
                rng, cond.temperature_k, cond.salinity_mm)
            axes.append(axis)
            radii.append(radius)
        analysis.load_precomputed(axes, radii)
        print(f"  synthesised {len(axes)} profiles (HOLE not called)")
    elif not hole_is_available():
        print(f"  HOLE executable '{HOLE_EXECUTABLE}' not found - skipping.")
        continue
    else:
        analysis.run()
        print(f"  profiled {len(analysis.radius)} conditions")

    # Register the profiles onto a common axis before any comparison.
    aligned_axes, shifts = align_profiles(analysis.axis, analysis.radius)
    print(f"  alignment shifts: min {shifts.min():+.2f} A, "
          f"max {shifts.max():+.2f} A, mean |shift| {np.abs(shifts).mean():.2f} A")

    pore[model_level] = {
        "analysis": analysis,
        "aligned_axes": aligned_axes,
        "shifts": shifts,
    }

print("\nDone.")

---

## 10. Constriction summary

In [ ]:
# Narrowest radius per condition - the number that decides whether an ion fits.
#
# A hydrated Na+ needs roughly 3.5 A of radius and a hydrated Ca2+ rather more,
# so a minimum radius below about 1.5 A is a closed gate on any reading.
for model_level, entry in pore.items():
    analysis = entry["analysis"]
    minima = analysis.minimum_radius()
    print(f"\n{'=' * 62}\n{model_level.upper()}\n{'=' * 62}")
    print(f"{'condition':<22}{'min radius (A)':>16}{'constrictions':>16}")
    print("-" * 62)
    for index, cond in enumerate(grid):
        n_constrictions = len(analysis.constrictions()[index][0])
        print(f"{str(cond):<22}{minima[index]:>16.2f}{n_constrictions:>16d}")

---

## 11. Profile plots

In [ ]:
# Aligned pore profiles, one panel per (model level, salinity), temperatures
# overlaid. The pore axis runs vertically so the figure reads like a cross
# section through the membrane.
for model_level, entry in pore.items():
    analysis = entry["analysis"]
    aligned_axes = entry["aligned_axes"]
    for salinity in SALINITIES_MM:
        indices = grid.where(salinity_mm=salinity)
        plot_pore_profiles(
            axes=[aligned_axes[i] for i in indices],
            radii=[analysis.radius[i] for i in indices],
            stds=[analysis.radius_std[i] for i in indices],
            labels=grid.labels(indices, by="temperature"),
            legend_title="Temperature",
            title=f"{model_level.title()} - {salinity:.0f} mM",
            vertical=True,
            show_band=True,
            figure_name=f"pore_profile_{model_level}_{salinity:.0f}mM",
        )
        plt.show()

---

## 12. Limiting constriction vs temperature

In [ ]:
# How the limiting constriction responds to temperature, one line per salinity.
for model_level, entry in pore.items():
    minima = entry["analysis"].minimum_radius()
    plot_minimum_radius(
        values_by_group=[minima[grid.where(salinity_mm=s)] for s in SALINITIES_MM],
        group_labels=[f"{s:.0f} mM" for s in SALINITIES_MM],
        x_values=list(TEMPERATURES_K),
        xlabel="Temperature (K)",
        title=f"{model_level.title()}: limiting constriction",
        figure_name=f"minimum_radius_{model_level}",
    )
    plt.show()

---

## 13. Pore surface export

In [ ]:
# Pore surface meshes for a representative condition at each model level, so the
# lumen can be inspected next to the structure in PyMOL or VMD.
REPRESENTATIVE_TEMPERATURE = 310
REPRESENTATIVE_SALINITY = 150

for model_level, entry in pore.items():
    index = grid.where(temperature_k=REPRESENTATIVE_TEMPERATURE,
                       salinity_mm=REPRESENTATIVE_SALINITY)[0]
    analysis = entry["analysis"]
    print(f"{model_level} ({grid[index]}):")
    write_pore_surface_pdb(
        entry["aligned_axes"][index],
        analysis.radius[index],
        OUTPUT_DIR / f"pore_surface_{model_level}_"
                     f"{REPRESENTATIVE_TEMPERATURE:.0f}K_"
                     f"{REPRESENTATIVE_SALINITY:.0f}mM.pdb",
    )

---

## Troubleshooting

**`RuntimeError: The HOLE executable ... was not found`**
HOLE is a separate program from `mdahole2`. Install it from
https://www.holeprogram.org/, put the binary on `PATH`, or set
`HOLE_EXECUTABLE` to its absolute path. To exercise everything else meanwhile,
set `USE_DEMO_DATA = True`.

**`HOLE returned no usable profile`**
Almost always a pore-axis problem. HOLE searches along a vector from a starting
point; if the channel is not aligned with z, or the start point is outside the
lumen, the search fails immediately. Set `HOLE_CVECT` to the pore axis and
`HOLE_CPOINT` to a point you know is inside the pore, then re-run.

**`The profiles share no common range of the reaction coordinate`**
The pathway moved a long way between frames, usually because the protein was not
centred, or drifted across a periodic boundary. Re-run `gmx trjconv` with
`-pbc mol -center`.

**Profiles run off into bulk solvent**
Lower `HOLE_END_RADIUS`. The default of 5 A stops when the pathway opens out;
raising it follows the vestibules further but can escape the protein entirely.

**Alignment shifts are implausibly large**
Check the reported shifts in the profiling cell. If they approach the length of
the pore, the constrictions being matched are probably not the same physical
features. Raise `CONSTRICTION_PROMINENCE` so only the deepest, most reliable
constrictions drive the alignment, or pass `max_shift` to `align_profiles`.

**It is unbearably slow**
HOLE runs once per frame. Raise `FRAME_STEP`; a stride giving 50-200 frames per
condition is usually plenty for a mean profile.

---

## Notes on method

- **Averaging.** Every frame's profile is resampled onto a common axis before
  averaging, using only the range all frames share. Averaging raw HOLE output
  element-wise is not safe: the sample points differ from frame to frame, so
  element *k* is not the same position along the pore in every frame.
- **Alignment is a registration step, not a physical correction.** It removes
  the arbitrary origin of HOLE's reaction coordinate. Always check the reported
  shifts; a large one means the matching is suspect.
- **The standard-deviation band** shows fluctuation across frames within one
  run, not uncertainty in the mean. A wide band means a mobile gate, which is
  itself a result.
- **Constriction detection** is peak-finding on the negated radius, so it
  reports local minima. It has no knowledge of which constriction is the
  selectivity filter and which is the gate; annotate those yourself using the
  positions and `DOMAIN_SEGMENTS`.
- **The pore surface PDB is a visualisation aid.** The pseudoatoms are placed on
  a circle of the local HOLE radius; a real lumen is not circular, so treat the
  mesh as indicative.

---

## Repository housekeeping

This notebook is one of several in the repository, so a few conventions keep
them from interfering with each other.

**Shared, repository-level files.** `requirements.txt` and `.gitignore` live at
the repository root, one of each, covering every notebook. Neither can be
renamed per notebook: git only recognises the filename `.gitignore`, and
`pip install -r requirements.txt` is what every reader and CI runner expects.

**Per-notebook outputs.** Each notebook writes to `results/<NOTEBOOK_SLUG>/`,
set in the configuration cell, so two notebooks saving a figure of the same name
cannot overwrite one another.

**Shared input data.** `DATA_ROOT` and `demo_data/` are deliberately *not*
namespaced, so sibling notebooks read the same trajectories rather than each
keeping a copy.

**Sections 1-4 of the configuration cell are identical in every notebook.** If
you change the simulation grid, the protomer count or the domain boundaries,
change them the same way in each notebook. If that becomes tedious, move those
sections into a `channel_config.py` at the repository root and import it from
each notebook instead.

**Outputs are stripped** from the committed notebooks, which keeps diffs small
and stops absolute data paths leaking through cell output:

```bash
pip install nbstripout
nbstripout --install     # run once, inside the repository
```
